# Annoy

> [Annoy](https://github.com/spotify/annoy) (`Approximate Nearest Neighbors Oh Yeah`) 是一个 C++ 库，带有 Python 绑定，用于搜索空间中与给定查询点接近的点。它还创建了大型的只读基于文件的数据库结构，这些结构被映射到内存中，以便多个进程可以共享相同的数据。

您需要安装 `langchain-community` (`pip install -qU langchain-community`) 才能使用此集成

本笔记本展示了如何使用与 `Annoy` 向量数据库相关的功能。

注意：Annoy 是只读的——一旦构建了索引，就无法再添加任何嵌入！
如果您想逐步向您的 VectorStore 添加新条目，那么最好选择其他方案！

In [ ]:
%pip install --upgrade --quiet  annoy

## 从文本创建向量存储

You can create a `VectorStore` from a list of texts.

您可以从文本列表中创建 `VectorStore`。

```python
from langchain.vectorstores import FAISS

# List of texts
texts = ["This is the first document.", "This document is the second document.", "And this is the third one.", "Is this the first document?"]

# Create a VectorStore from the texts
vector_store = FAISS.from_texts(texts, embeddings=None)
```

```python
from langchain.vectorstores import Chroma

# List of texts
texts = ["This is the first document.", "This document is the second document.", "And this is the third one.", "Is this the first document?"]

# Create a VectorStore from the texts
vector_store = Chroma.from_texts(texts, embeddings=None)
```

You can also create a `VectorStore` from a list of documents.

您也可以从文档列表中创建 `VectorStore`。

```python
from langchain.schema import Document
from langchain.vectorstores import FAISS

# List of documents
documents = [
    Document(page_content="This is the first document.", metadata={"source": "doc1"}),
    Document(page_content="This document is the second document.", metadata={"source": "doc2"}),
    Document(page_content="And this is the third one.", metadata={"source": "doc3"}),
    Document(page_content="Is this the first document?", metadata={"source": "doc4"}),
]

# Create a VectorStore from the documents
vector_store = FAISS.from_documents(documents, embeddings=None)
```

```python
from langchain.schema import Document
from langchain.vectorstores import Chroma

# List of documents
documents = [
    Document(page_content="This is the first document.", metadata={"source": "doc1"}),
    Document(page_content="This document is the second document.", metadata={"source": "doc2"}),
    Document(page_content="And this is the third one.", metadata={"source": "doc3"}),
    Document(page_content="Is this the first document?", metadata={"source": "doc4"}),
]

# Create a VectorStore from the documents
vector_store = Chroma.from_documents(documents, embeddings=None)
```

You can also specify an embeddings object to use for creating the VectorStore.

您还可以指定一个 embeddings 对象来用于创建 VectorStore。

```python
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# List of texts
texts = ["This is the first document.", "This document is the second document.", "And this is the third one.", "Is this the first document?"]

# Create an embeddings object
embeddings = OpenAIEmbeddings()

# Create a VectorStore from the texts with embeddings
vector_store = FAISS.from_texts(texts, embeddings=embeddings)
```

```python
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma

# List of texts
texts = ["This is the first document.", "This document is the second document.", "And this is the third one.", "Is this the first document?"]

# Create an embeddings object
embeddings = OpenAIEmbeddings()

# Create a VectorStore from the texts with embeddings
vector_store = Chroma.from_texts(texts, embeddings=embeddings)

In [ ]:
from langchain_community.vectorstores import Annoy
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings_func = HuggingFaceEmbeddings(model_name=model_name)

In [4]:
texts = ["pizza is great", "I love salad", "my car", "a dog"]

# default metric is angular
vector_store = Annoy.from_texts(texts, embeddings_func)

In [4]:
# allows for custom annoy parameters, defaults are n_trees=100, n_jobs=-1, metric="angular"
vector_store_v2 = Annoy.from_texts(
    texts, embeddings_func, metric="dot", n_trees=100, n_jobs=1
)

In [5]:
vector_store.similarity_search("food", k=3)

[Document(page_content='pizza is great', metadata={}),
 Document(page_content='I love salad', metadata={}),
 Document(page_content='my car', metadata={})]

In [6]:
# the score is a distance metric, so lower is better
vector_store.similarity_search_with_score("food", k=3)

[(Document(page_content='pizza is great', metadata={}), 1.0944390296936035),
 (Document(page_content='I love salad', metadata={}), 1.1273186206817627),
 (Document(page_content='my car', metadata={}), 1.1580758094787598)]

## 从文档创建 VectorStore

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("../../how_to/state_of_the_union.txtn.txtn.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

In [8]:
docs[:5]

[Document(page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.', metadata={'source'

In [9]:
vector_store_from_docs = Annoy.from_documents(docs, embeddings_func)

In [10]:
query = "What did the president say about Ketanji Brown Jackson"
docs = vector_store_from_docs.similarity_search(query)

In [11]:
print(docs[0].page_content[:100])

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Ac


## 通过现有嵌入创建 VectorStore

In [12]:
embs = embeddings_func.embed_documents(texts)

In [13]:
data = list(zip(texts, embs))

vector_store_from_embeddings = Annoy.from_embeddings(data, embeddings_func)

In [14]:
vector_store_from_embeddings.similarity_search_with_score("food", k=3)

[(Document(page_content='pizza is great', metadata={}), 1.0944390296936035),
 (Document(page_content='I love salad', metadata={}), 1.1273186206817627),
 (Document(page_content='my car', metadata={}), 1.1580758094787598)]

## 通过嵌入进行搜索

This guide shows you how to perform semantic search on your data using embeddings.

本指南将向您展示如何使用嵌入对数据执行语义搜索。

Semantic search allows you to find information based on the meaning of your query, rather than just keywords. For example, if you search for "how to bake a cake", a semantic search might return results about "cake recipes" or "baking instructions", even if the exact keywords aren't present.

语义搜索允许您根据查询的含义查找信息，而不仅仅是关键词。例如，如果您搜索“如何烤蛋糕”，即使没有完全匹配的关键词，语义搜索也可能返回关于“蛋糕食谱”或“烘焙说明”的结果。

### How it works

### 工作原理

1.  **Generate Embeddings**: You convert your data (text, images, etc.) into numerical representations called embeddings using an embedding model. These embeddings capture the semantic meaning of the data.

    1.  **生成嵌入**: 您使用嵌入模型将数据（文本、图像等）转换为称为嵌入的数值表示。这些嵌入捕获了数据的语义含义。

2.  **Store Embeddings**: You store these embeddings in a vector database. A vector database is optimized for storing and querying high-dimensional vectors.

    2.  **存储嵌入**: 您将这些嵌入存储在向量数据库中。向量数据库经过优化，可以存储和查询高维向量。

3.  **Query Embeddings**: When a user performs a search, their query is also converted into an embedding using the same model.

    3.  **查询嵌入**: 当用户执行搜索时，他们的查询也会使用相同的模型转换为嵌入。

4.  **Vector Search**: The vector database then finds the embeddings in its storage that are most similar to the query embedding. Similarity is typically measured using metrics like cosine similarity or Euclidean distance.

    4.  **向量搜索**: 然后，向量数据库在其存储中查找与查询嵌入最相似的嵌入。相似性通常使用余弦相似度或欧几里得距离等指标来衡量。

5.  **Retrieve Results**: The data corresponding to the most similar embeddings are retrieved and presented to the user as search results.

    5.  **检索结果**: 与最相似的嵌入对应的数据将被检索出来，并作为搜索结果呈现给用户。

### Example using LangChain

### 使用 LangChain 的示例

LangChain provides tools to easily integrate embedding models and vector databases for semantic search.

LangChain 提供了易于集成的工具，用于嵌入模型和向量数据库以进行语义搜索。

```python
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Sample data
documents = [
    Document(page_content="The quick brown fox jumps over the lazy dog."),
    Document(page_content="A journey of a thousand miles begins with a single step."),
    Document(page_content="To be or not to be, that is the question."),
    Document(page_content="The early bird catches the worm."),
]

# 1. Generate Embeddings and store them in a vector database
# Initialize the embedding model (e.g., OpenAI's text-embedding-ada-002)
# Ensure you have your OpenAI API key set as an environment variable OPENAI_API_KEY
embeddings = OpenAIEmbeddings()

# Create a vector store from the documents
# FAISS is a library for efficient similarity search and clustering of dense vectors
vectorstore = FAISS.from_documents(documents, embeddings)

# 2. Perform a semantic search
query = "What is the proverb about starting a long journey?"

# Find documents similar to the query
# k specifies the number of documents to return
similar_docs = vectorstore.similarity_search(query, k=1)

# Print the most similar document
print(f"Query: {query}")
print("\nMost similar document:")
for doc in similar_docs:
    print(f"- {doc.page_content}")

```

**Explanation**:

**说明**:

1.  **`OpenAIEmbeddings()`**: This initializes an embedding model from OpenAI. You need to have your OpenAI API key configured (usually as an environment variable `OPENAI_API_KEY`).

    1.  **`OpenAIEmbeddings()`**: 这初始化了 OpenAI 的嵌入模型。您需要配置您的 OpenAI API 密钥（通常设置为环境变量 `OPENAI_API_KEY`）。

2.  **`FAISS.from_documents(documents, embeddings)`**: This takes your list of `Document` objects and the `embeddings` model, generates embeddings for each document, and stores them in a FAISS index (an efficient in-memory vector store).

    2.  **`FAISS.from_documents(documents, embeddings)`**: 这接受您的 `Document` 对象列表和 `embeddings` 模型，为每个文档生成嵌入，并将它们存储在 FAISS 索引（一个高效的内存中向量存储）中。

3.  **`vectorstore.similarity_search(query, k=1)`**: This converts the `query` string into an embedding using the same `embeddings` model and then searches the `vectorstore` for the document whose embedding is most similar to the query embedding. `k=1` means it will return the single most similar document.

    3.  **`vectorstore.similarity_search(query, k=1)`**: 这使用相同的 `embeddings` 模型将 `query` 字符串转换为嵌入，然后在 `vectorstore` 中搜索其嵌入与查询嵌入最相似的文档。`k=1` 表示它将返回最相似的单个文档。

This example demonstrates a basic semantic search. You can extend this by:

此示例演示了基本的语义搜索。您可以对其进行扩展：

*   Using different embedding models (e.g., from Hugging Face, Cohere).
*   Using different vector databases (e.g., Chroma, Pinecone, Weaviate).
*   Implementing more complex retrieval strategies (e.g., Maximum Marginal Relevance - MMR).
*   Integrating this into a larger application, like a question-answering system or a recommendation engine.

*   使用不同的嵌入模型（例如，来自 Hugging Face、Cohere）。
*   使用不同的向量数据库（例如，Chroma、Pinecone、Weaviate）。
*   实现更复杂的检索策略（例如，最大边际相关性 - MMR）。
*   将此集成到更大的应用程序中，例如问答系统或推荐引擎。

In [15]:
motorbike_emb = embeddings_func.embed_query("motorbike")

In [16]:
vector_store.similarity_search_by_vector(motorbike_emb, k=3)

[Document(page_content='my car', metadata={}),
 Document(page_content='a dog', metadata={}),
 Document(page_content='pizza is great', metadata={})]

In [17]:
vector_store.similarity_search_with_score_by_vector(motorbike_emb, k=3)

[(Document(page_content='my car', metadata={}), 1.0870471000671387),
 (Document(page_content='a dog', metadata={}), 1.2095637321472168),
 (Document(page_content='pizza is great', metadata={}), 1.3254905939102173)]

## 通过 docstore id 搜索

In [18]:
vector_store.index_to_docstore_id

{0: '2d1498a8-a37c-4798-acb9-0016504ed798',
 1: '2d30aecc-88e0-4469-9d51-0ef7e9858e6d',
 2: '927f1120-985b-4691-b577-ad5cb42e011c',
 3: '3056ddcf-a62f-48c8-bd98-b9e57a3dfcae'}

In [19]:
some_docstore_id = 0  # texts[0]

vector_store.docstore._dict[vector_store.index_to_docstore_id[some_docstore_id]]

Document(page_content='pizza is great', metadata={})

In [20]:
# same document has distance 0
vector_store.similarity_search_with_score_by_index(some_docstore_id, k=3)

[(Document(page_content='pizza is great', metadata={}), 0.0),
 (Document(page_content='I love salad', metadata={}), 1.0734446048736572),
 (Document(page_content='my car', metadata={}), 1.2895267009735107)]

## 保存和加载

You can save and load your work in a few ways.

您可以通过几种方式保存和加载您的工作。

### Saving your work

### 保存您的工作

To save your work, click the **Save** button in the top-right corner of the editor.

要保存您的工作，请点击编辑器右上角的**保存**按钮。

This will save your current project to your account.

这将把您当前的项目保存到您的账户中。

### Loading your work

### 加载您的工作

To load your work, click the **Load** button in the top-right corner of the editor.

要加载您的工作，请点击编辑器右上角的**加载**按钮。

This will open a list of your saved projects. You can then select a project to load.

这将打开一个您已保存项目的列表。然后您可以选择一个项目进行加载。

### Saving and loading with code

### 使用代码保存和加载

You can also save and load your work using code.

您也可以使用代码保存和加载您的工作。

To save your work, use the `save` function:

要保存您的工作，请使用 `save` 函数：

```javascript
save();
```

To load your work, use the `load` function:

要加载您的工作，请使用 `load` 函数：

```javascript
load();

In [21]:
vector_store.save_local("my_annoy_index_and_docstore")

saving config


In [22]:
loaded_vector_store = Annoy.load_local(
    "my_annoy_index_and_docstore", embeddings=embeddings_func
)

In [23]:
# same document has distance 0
loaded_vector_store.similarity_search_with_score_by_index(some_docstore_id, k=3)

[(Document(page_content='pizza is great', metadata={}), 0.0),
 (Document(page_content='I love salad', metadata={}), 1.0734446048736572),
 (Document(page_content='my car', metadata={}), 1.2895267009735107)]

## 从头开始构建

In [25]:
import uuid

from annoy import AnnoyIndex
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document

metadatas = [{"x": "food"}, {"x": "food"}, {"x": "stuff"}, {"x": "animal"}]

# embeddings
embeddings = embeddings_func.embed_documents(texts)

# embedding dim
f = len(embeddings[0])

# index
metric = "angular"
index = AnnoyIndex(f, metric=metric)
for i, emb in enumerate(embeddings):
    index.add_item(i, emb)
index.build(10)

# docstore
documents = []
for i, text in enumerate(texts):
    metadata = metadatas[i] if metadatas else {}
    documents.append(Document(page_content=text, metadata=metadata))
index_to_docstore_id = {i: str(uuid.uuid4()) for i in range(len(documents))}
docstore = InMemoryDocstore(
    {index_to_docstore_id[i]: doc for i, doc in enumerate(documents)}
)

db_manually = Annoy(
    embeddings_func.embed_query, index, metric, docstore, index_to_docstore_id
)

In [26]:
db_manually.similarity_search_with_score("eating!", k=3)

[(Document(page_content='pizza is great', metadata={'x': 'food'}),
  1.1314140558242798),
 (Document(page_content='I love salad', metadata={'x': 'food'}),
  1.1668788194656372),
 (Document(page_content='my car', metadata={'x': 'stuff'}), 1.226445198059082)]